# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata.to_json()
# Print dataset name and description from metadata
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets present by @id
record_sets = dataset.record_sets()
print(f"Number of record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    print(f"  Fields:")
    for field in rs.get('fields', []):
        print(f"    Field @id: {field['@id']} | Name: {field.get('name', 'N/A')}")
    print()
# Optionally, print the first record for each record set
for rs in record_sets:
    rs_id = rs['@id']
    try:
        first_records = list(dataset.records(record_set=rs_id))[:1]
        if first_records:
            print(f"First record in record set {rs_id}:")
            print(first_records[0])
        else:
            print(f"No records found in record set {rs_id}.")
    except Exception as e:
        print(f"Error fetching records for {rs_id}: {e}")
    print()

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set

# Collect @id values for record sets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[rs_id] = df

# Display columns of the first record set
if record_set_ids:
    print(f"Columns in record set {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    dataframes[record_set_ids[0]].head()
else:
    print("No record sets available")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section can include removing outliers, transforming distributions, or grouping by key attributes.

In [ ]:
# Choose a record set with numeric fields for EDA

# Find a record set with at least one numeric column
numeric_rs_id = None
numeric_field_id = None
group_field_id = None
for rs in record_sets:
    rs_id = rs['@id']
    df = dataframes[rs_id]
    if not df.empty:
        # Look for float/int columns
        for field in rs.get('fields', []):
            if field.get('dataType') in ['Float', 'Integer', 'Number']:
                col_id = field['@id']
                if col_id in df.columns:
                    numeric_rs_id = rs_id
                    numeric_field_id = col_id
                    break
        # For grouping, pick first field with limited unique values
        for field in rs.get('fields', []):
            col_id = field['@id']
            if col_id in df.columns and df[col_id].nunique() < 5:
                group_field_id = col_id
        if numeric_rs_id:
            break

if numeric_rs_id and numeric_field_id:
    print(f"Selected numeric field: {numeric_field_id} from record set {numeric_rs_id}")
    df = dataframes[numeric_rs_id]
    # Remove any missing values for this column
    df = df.dropna(subset=[numeric_field_id])
    # Try to convert the column to numeric dtype
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Quick visualization of numeric field distribution
if numeric_rs_id and numeric_field_id:
    df = dataframes[numeric_rs_id]
    df = df.dropna(subset=[numeric_field_id])
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a group_field is available, show boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset package provides structured records for clinical features, hormone levels, imaging, and genetic variants for non-classical 21-hydroxylase deficiency infertility.
- Records and fields are referenced via their `@id`, ensuring robust and reproducible data extraction and manipulation.
- Data was extracted and visualized with `mlcroissant`, demonstrating filtering, normalization, and grouping by field.
- Limitations include small sample size and unique cohort (N=3).
- Recommend the dataset for academic research and case review, not direct ML development.
- Please refer to full Croissant metadata for further details and entity relationships.